# Task 1 population failure playground

A Task 1 submission is a **population of cells**, not one average expression vector. This notebook deliberately breaks the known public E9.5 target in a few controlled ways so you can see which parts of the score care. These are target-aware teaching controls, not a benchmark.

The experiment definitions live in [`experiments.py`](experiments.py). The notebook and the committed result generator call the same functions, so there is only one implementation to keep in sync.

In [ ]:
!pip -q install "git+https://github.com/aristoteleo/veckit.git@46d41e63f42a9aab815db20b742feeccd249cb17" pandas matplotlib

In [ ]:
from pathlib import Path
import sys, urllib.request
import anndata as ad
import numpy as np
import pandas as pd

LAB = Path.cwd() / 'intuition-lab'
if not (LAB / 'experiments.py').exists():
    LAB = Path.cwd()
sys.path.insert(0, str(LAB))
from experiments import dense, make_t1_failures, score_t1_failures

DATA = Path('/content/vec_t1_failures')
DATA.mkdir(exist_ok=True)
base = 'https://raw.githubusercontent.com/aristoteleo/veckit/46d41e63f42a9aab815db20b742feeccd249cb17/data/'
for name in ['sample_8.5.h5ad', 'sample_9.5.h5ad']:
    p = DATA / name
    if not p.exists(): urllib.request.urlretrieve(base + name, p)

ref_path = DATA / 'sample_8.5.h5ad'
target_path = DATA / 'sample_9.5.h5ad'
target = ad.read_h5ad(target_path)
print('target:', target.shape, '| celltype labels:', 'celltype' in target.obs)

## Break one thing at a time

- **row_order_only** reorders cells. A population should not care about row order.
- **repeated_mean** makes every cell equal to the target mean. Pseudobulk is perfect, diversity is gone.
- **gene_wise_shuffle** preserves each gene's marginal distribution but destroys which genes co-occur inside a cell.
- **one_state_only** resamples a single observed cell type until it fills the whole population. The individual cells are real-looking; the mixture is wrong.

In [ ]:
preds, meta = make_t1_failures(target, DATA / 'predictions', seed=0)
print('built:', list(preds))
if meta['one_state_label'] is not None:
    print('one_state_only uses:', meta['one_state_label'], f"({meta['one_state_pool']} source cells)")

In [ ]:
X = dense(target)
rows = []
for name, path in preds.items():
    Y = dense(ad.read_h5ad(path))
    rows.append({
        'prediction': name,
        'mean_abs_error': float(np.mean(np.abs(Y.mean(0) - X.mean(0)))),
        'variance_abs_error': float(np.mean(np.abs(Y.var(0) - X.var(0)))),
    })
pd.DataFrame(rows).set_index('prediction').round(5)

In [ ]:
scores = score_t1_failures(preds, target=target_path, reference=ref_path)
scores[['pseudobulk_pearson', 'mmd_u', 'variogram', 'variance_ratio', 'composition_JSD']].round(4)

## What I would remember

The repeated-mean control is the useful one to keep in your head. It can nail the aggregate expression while being an obviously terrible population model. If a method looks good only through pseudobulk-style summaries, check whether it has actually learned heterogeneity and mixture structure.